In [6]:
import openpyxl
import pandas as pd
import os

def clean_estimate_sheet(file_path, sheet_name):
    wb = openpyxl.load_workbook(file_path, data_only=True)
    if sheet_name not in wb.sheetnames:
        return None, f"Лист '{sheet_name}' не найден."
    
    sheet = wb[sheet_name]
    header_row_idx = None
    
    # 1. Поиск строки заголовка по ключевым словам
    for i, row in enumerate(sheet.iter_rows(min_row=1, max_row=min(50, sheet.max_row), values_only=True)):
        row_vals = [str(cell).strip() for cell in row if cell is not None]
        row_str = " ".join(row_vals)
        
        # Ищем строку, где есть № п/п и Наименование
        if "№ п/п" in row_str and ("Наименование" in row_str or "работ" in row_str.lower()):
            header_row_idx = i + 1
            break
            
    if header_row_idx is None:
        return None, "Заголовок таблицы не найден."
    
    # 2. Формируем чистые заголовки вручную, избегая проблем с pandas internals
    raw_headers = list(sheet.iter_rows(min_row=header_row_idx, max_row=header_row_idx, values_only=True))[0]
    
    # Убираем дубликаты и пустые значения, создавая уникальные имена
    seen = set()
    clean_headers = []
    for h in raw_headers:
        val = str(h).strip() if h is not None else ""
        if not val or val == "None":
            continue
            
        # Если такое имя уже было, добавляем суффикс для уникальности
        base_name = val
        counter = 1
        unique_name = base_name
        while unique_name in seen:
            unique_name = f"{base_name}_{counter}"
            counter += 1
            
        seen.add(unique_name)
        clean_headers.append(unique_name)
        
        # Ограничиваем количество столбцов разумным пределом (в сметах обычно до 15-20 значимых)
        if len(clean_headers) > 30: 
            break

    # 3. Читаем данные, используя usecols для ограничения ширины таблицы
    # Это предотвращает захват бесконечных пустых или дублирующихся столбцов справа
    num_cols = len(clean_headers)
    
    df = pd.read_excel(
        file_path, 
        sheet_name=sheet_name, 
        skiprows=header_row_idx, 
        header=None,
        usecols=range(num_cols),  # <-- Важно: читаем только нужное количество столбцов
        nrows=5000  # Опционально: лимит строк для скорости
    )
    
    # Присваиваем наши очищенные заголовки
    if len(df.columns) == len(clean_headers):
        df.columns = clean_headers
    else:
        # Если количество не совпало, обрезаем заголовки под реальное число столбцов
        df.columns = clean_headers[:len(df.columns)]
        
    # Удаляем мусор: полностью пустые строки и строки, которые являются продолжением шапки
    df = df.dropna(how='all')
    
    # Фильтр против служебных строк с номерами колонок (1, 2, 3...)
    if len(df) > 0:
        first_col = df.iloc[:, 0].astype(str).str.strip()
        df = df[~first_col.str.match(r'^\d+$', na=False)]
        
    return df.reset_index(drop=True), f"OK (строка {header_row_idx}, стлбцов: {num_cols})"

In [ ]:
# === ЗАПУСК ===
file_path = "Локально сметный расчет.xlsx"
output_file = "Очищенные_сметы_v2.xlsx"

if os.path.exists(file_path):
    wb = openpyxl.load_workbook(file_path, data_only=True)
    results = {}
    
    print(f"Обработка {len(wb.sheetnames)} листов...")
    for s_name in wb.sheetnames:
        if s_name in ['Дефлятор', 'Титульный']: 
            continue
            
        df, msg = clean_estimate_sheet(file_path, s_name)
        print(f"[{s_name}] {msg}")
        if df is not None:
            results[s_name] = df
            
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        for name, data in results.items():
            data.to_excel(writer, sheet_name=name[:31], index=False) # Excel лимит имени листа 31 символ
            
    print(f"\n✅ Сохранено в: {output_file}")
else:
    print("Файл не найден!")
            
    # Сохраняем очищенные данные в новый файл
    # output_file = "Очищенные_сметы.xlsx"
    # with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    #     for sheet_name, df in all_clean_data.items():
    #         df.to_excel(writer, sheet_name=sheet_name, index=False)
    # print(f"\n✅ Готово! Очищенные данные сохранены в файл: {output_file}")

Обработка 10 листов...
[ССРСС] OK (строка 20, стлбцов: 8)
[ОС-02-01] OK (строка 16, стлбцов: 8)
[ЛС02-01-01.АР] OK (строка 30, стлбцов: 12)
[ЛС02-01-02.ВК] OK (строка 30, стлбцов: 12)
[ЛС02-01-03.Вентиляция] OK (строка 30, стлбцов: 12)
[ЛС02-01-04.Отопление] OK (строка 30, стлбцов: 12)
[ЛС02-01-05.ЭОМ] OK (строка 30, стлбцов: 12)
[ЛС02-01-06.АПС СОУЭ] OK (строка 30, стлбцов: 12)
[СР-01] OK (строка 30, стлбцов: 12)

✅ Сохранено в: Очищенные_сметы_v2.xlsx


In [10]:
results['ССРСС']

,№ п/п,Обоснование,"Наименование локальных сметных расчётов (смет), затрат","Сметная стоимость, тыс. руб.",№ п/п_1,Обоснование_1,"Наименование локальных сметных расчётов (смет), затрат_1","Сметная стоимость, тыс. руб._1"
0,NaN,NaN,NaN,"Строительных (ремонтно-строительных, ремонтно-...",монтажных работ,оборудования,прочих затрат,всего
1,"Глава 2. ОСНОВНЫЕ ОБЪЕКТЫ СТРОИТЕЛЬСТВА, РЕКОН...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,Итого по Главе 2,47382.57,7901.29,3492.74,NaN,58776.6
3,NaN,NaN,Итого по Главам 1-7,47382.57,7901.29,3492.74,NaN,58776.6
4,NaN,NaN,Итого по Главам 1-8,47382.57,7901.29,3492.74,NaN,58776.6
5,Глава 9. ПРОЧИЕ РАБОТЫ И ЗАТРАТЫ,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,Итого по Главе 9,668.09,111.41,NaN,197.54,977.04
7,NaN,NaN,Итого по Главам 1-9,48050.66,8012.7,3492.74,197.54,59753.64
8,NaN,NaN,Итого по Главам 1-10,48050.66,8012.7,3492.74,197.54,59753.64
9,NaN,NaN,Итого по Главам 1-11,48050.66,8012.7,3492.74,197.54,59753.64
